In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# Single Kaggle ARC-AGI-3 test cell
# Uses LOCAL competition wheels, visible terminal playback, and ACTION5 as detach/interact.

from pathlib import Path
import sys
import subprocess
import json
import time
from typing import Any

# ---------------------------------
# 1) Offline install from Kaggle input
# ---------------------------------
COMP_ROOT = Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3")
WHEEL_DIR = COMP_ROOT / "arc_agi_3_wheels"

if not WHEEL_DIR.exists():
    raise FileNotFoundError(f"Wheel directory not found: {WHEEL_DIR}")

subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "--no-index",
    f"--find-links={WHEEL_DIR}",
    "arc_agi",
])

# ---------------------------------
# 2) Imports after local install
# ---------------------------------
import arc_agi
from arcengine import GameAction

# ---------------------------------
# 3) Locked mapping
# ---------------------------------
GAME_ID = "ls20"

# Current empirical mapping
# ACTION1 = up
# ACTION3 = right
# ACTION4 = left
# ACTION5 = interact / detach
ACTION_MAP = {
    "U": GameAction.ACTION1,
    "R": GameAction.ACTION3,
    "L": GameAction.ACTION4,
    "I": GameAction.ACTION5,
}

# Exact locked route, with all old E/release steps replaced by I/interact
SCRIPT = """
R R R U U U U
L L L U U U U
L U U U U
R L
U U
L L L
I I I I I I I I
U U
L
U U U
R R R
U U U
R R R
I I
R
I I I I I I I I I I I
""".split()

TRACE_PATH = Path("/kaggle/working/arcagi3_action5_trace.jsonl")
SCORECARD_PATH = Path("/kaggle/working/arcagi3_action5_scorecard.json")

def make_jsonable(x: Any, depth: int = 0, max_depth: int = 4) -> Any:
    if x is None or isinstance(x, (str, int, float, bool)):
        return x
    if depth >= max_depth:
        return f"<max_depth:{type(x).__name__}>"
    if isinstance(x, dict):
        return {str(k): make_jsonable(v, depth + 1, max_depth) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return [make_jsonable(v, depth + 1, max_depth) for v in x]
    if hasattr(x, "model_dump"):
        try:
            return make_jsonable(x.model_dump(), depth + 1, max_depth)
        except Exception:
            pass
    if hasattr(x, "dict"):
        try:
            return make_jsonable(x.dict(), depth + 1, max_depth)
        except Exception:
            pass
    return repr(x)

def extract_state(step_result: Any) -> str:
    if isinstance(step_result, tuple):
        if len(step_result) >= 5:
            _, _, terminated, truncated, info = step_result[:5]
            if isinstance(info, dict):
                for key in ("state", "game_state", "status"):
                    if key in info:
                        return str(info[key])
            if terminated:
                return "TERMINATED"
            if truncated:
                return "TRUNCATED"
        elif len(step_result) >= 4:
            _, _, done, info = step_result[:4]
            if isinstance(info, dict):
                for key in ("state", "game_state", "status"):
                    if key in info:
                        return str(info[key])
            if done:
                return "DONE"
    return "UNKNOWN"

print("[CONFIG]")
print("GAME_ID:", GAME_ID)
print("TOKENS :", " ".join(SCRIPT))
print("MAP    :", {k: v.name for k, v in ACTION_MAP.items()})

# ---------------------------------
# 4) Visible playback
# ---------------------------------
arc = arc_agi.Arcade()
env = arc.make(GAME_ID, render_mode="terminal")

if hasattr(env, "reset"):
    try:
        env.reset()
    except TypeError:
        pass

trace_rows = []

print("\n[START BOARD]")
time.sleep(1.0)

print("\n[PLAYBACK START]")
for i, token in enumerate(SCRIPT, start=1):
    action = ACTION_MAP[token]
    result = env.step(action)
    state = extract_state(result)

    row = {
        "step": i,
        "token": token,
        "action": action.name,
        "state": state,
    }
    trace_rows.append(row)

    print(f"[{i:02d}/{len(SCRIPT):02d}] token={token} action={action.name} state={state}")

    if token == "I":
        time.sleep(0.70)
    else:
        time.sleep(0.20)

# ---------------------------------
# 5) Save outputs
# ---------------------------------
scorecard = arc.get_scorecard()

TRACE_PATH.write_text(
    "\n".join(json.dumps(r) for r in trace_rows) + "\n",
    encoding="utf-8",
)

SCORECARD_PATH.write_text(
    json.dumps(make_jsonable(scorecard), indent=2),
    encoding="utf-8",
)

print("\n[TRACE FILE]")
print(TRACE_PATH)

print("\n[SCORECARD FILE]")
print(SCORECARD_PATH)

print("\n[SCORECARD]")
print(json.dumps(make_jsonable(scorecard), indent=2))

if hasattr(env, "close"):
    env.close()